In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input, Normalization
import matplotlib.pyplot as plt

from adaptive_latents import datasets
import numpy as np
from adaptive_latents.utils import resample_matched_timeseries


In [ ]:
d = datasets.Odoherty21Dataset(bin_width=0.03)  # (timesteps x neurons)
beh = resample_matched_timeseries(d.behavioral_data, d.behavioral_data.t, d.neural_data.t)  # (timesteps x 2)

normalized_beh = (beh - np.nanmean(beh, axis=0)) / np.nanstd(beh, axis=0)

s = np.isnan(beh).any(axis=1) | np.isnan(d.neural_data).any(axis=1)
neural_data, beh = d.neural_data[~s], beh[~s]

In [ ]:
n = Normalization()
n.adapt(beh[None,:,:])
n.weights

In [ ]:
model = Sequential(
    [
        Input(shape=neural_data.shape),
        LSTM(200, return_sequences=True),
        Dense(2),
    ]
)

model.compile(optimizer='adam', loss='mse')
model.fit(neural_data[None, :,:], normalized_beh[None,:,:], epochs=100, verbose=True)

In [ ]:
%matplotlib qt

pred = model.predict(neural_data[None,:,:]).squeeze()
plt.scatter(beh.flatten(), pred.flatten(), s=5, c=np.arange(pred.size), cmap='plasma')


In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=5).fit(neural_data)
pca.components_[0,:]

preturbed_neural_data = neural_data.copy()
preturbed_neural_data[preturbed_neural_data.time_to_sample(500): preturbed_neural_data.time_to_sample(560),:] += pca.components_[0,:] * 2
# np.isnan(preturbed_neural_data).any()


preturbed_pred = model.predict(preturbed_neural_data[None,:,:]).squeeze()


In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
plt.plot(beh.t, beh[:,0])
plt.plot(beh.t, (pred*beh.std(axis=0) + beh.mean(axis=0))[:,0])
plt.plot(beh.t, (preturbed_pred*beh.std(axis=0) + beh.mean(axis=0))[:,0])

plt.xlim([500, 560])
